# Login

In [1]:
import time
from dotenv import load_dotenv
from selenium import webdriver
from Service.DayForce.Utilites.selenium_helper import handywrapper
import os
load_dotenv()
class login_dayforce:
    def login(self):
        company_id = os.environ['COMPANY_ID']
        username = os.environ['USER_NAME']
        password = os.environ['PASSWORD']
        host_address = os.environ['HOST_ADDRESS']

        driver=webdriver.Chrome()
        driver.get(host_address)
        selenium_helper = handywrapper(driver)
        
        
        # dayforce company id
        selenium_helper.send_keys_to_element("name", 'ctl00$MainContent$loginUI$txtCompanyId', company_id)
        selenium_helper.click_element("xpath", "//input[@value='Continue to username']")

        # dayforce account username
        selenium_helper.send_keys_to_element("name", 'ctl00$MainContent$loginUI$txtNewUserName', username)
        selenium_helper.click_element("xpath", "//input[@value='Continue to password']")

        time.sleep(1)
        # dayforce account password
        selenium_helper.send_keys_to_element("name", 'ctl00$MainContent$loginUI$txtNewUserPass', password)
        selenium_helper.click_element("xpath", "//input[@value='Log in']")


        time.sleep(5)
        # skip account recovery popup
        selenium_helper.click_element("xpath", "//button[@aria-label='Close']")
        selenium_helper.click_element("xpath", "//span[@widgetid='Button_2']")

        # login as status test001
        selenium_helper.click_element("xpath", "//span[contains(text(),'Next')]")
        return driver

In [2]:
login = login_dayforce()
driver=login.login()

# GET and Post Methods


In [ ]:
import re
import requests
class Pay_Category_Get_Post:
    def get_scrub_id(self, driver):
        current_url=driver.current_url
        scrub_id = re.search(r'/u/([^/]+)/', current_url).group(1)
        return scrub_id
    
    
    def get_session_data(self, driver):
        session = requests.Session()
        for cookie in driver.get_cookies():
            session.cookies.set(
                cookie["name"],
                cookie["value"]
            )
        csrf = driver.execute_script("""
        return window.Dayforce?.AppSettingsData?.csrfRequestToken;
        """)
        return session, csrf
    

    def update_pay_category(self, driver,payload):
        session, csrf = self.get_session_data(driver)
        scrub_id = self.get_scrub_id(driver)

        url = f"https://usstage261.dayforcehcm.com/MyDayforce/u/{scrub_id}/WFMAdmin/PayCategory/PersistPayCategory"
        headers = {
            "Accept": "*/*",
            "Content-Type": "application/json",
            "X-CSRF-Token": csrf,
            "X-Requested-With": "XMLHttpRequest",
            "Origin": "https://usstage261.dayforcehcm.com",
            "Referer": f"https://usstage261.dayforcehcm.com/MyDayforce/u/{scrub_id}/Common/",
        }

        response = session.post(
            url,
            json=payload,
            headers=headers
        )
        if response.status_code == 200:
            print("object updated successfully")
        else:
            print(f"Error: {response.status_code}")

    def post_pay_category(self, api_url, driver):
        session = requests.Session()
        for cookie in driver.get_cookies():
            session.cookies.set(cookie["name"], cookie["value"])
        response = session.post(api_url)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error: {response.status_code}")
            return None

# Model

In [5]:
class Pay_Category_Model:
    item_id=''
    pay_category_id=''
    processing_status=''
    object_name=''
    object_description=''
    pay_category_group=''
    multiplier_rate=''
    category=''
    show_hours=''
    show_amount=''
    is_irregular_cost=''
    sort_order=''
    code_name=''
    reference_code=''
    reference_code_2=''


# Reverse

In [136]:
class Pay_Category_Reverse:
    def Pay_Category_Reverse(self,driver):
        pay_category_api = Pay_Category_Get_Post()
        Scrub_id = pay_category_api.get_scrub_id(driver)

        url_pay_category = f"https://usstage261.dayforcehcm.com/MyDayforce/u/{Scrub_id}/Common/#SFRNTFBheUNhdGVnb3J5"
        driver.get(url_pay_category)
        pay_categories_api_url = f"https://usstage261.dayforcehcm.com/MyDayforce/u/{Scrub_id}/WFMAdmin/PayCategory/GetPayCategories"
        pay_categories_group_api_url = f"https://usstage261.dayforcehcm.com/MyDayforce/u/{Scrub_id}/WFMAdmin/PayCategory/GetPayCategoryGroups"

        pay_categories_data=pay_category_api.post_pay_category(pay_categories_api_url, driver)['EntityLists'][0]['Entities']

        group_data=pay_category_api.post_pay_category(pay_categories_group_api_url, driver)
        group_id_to_name={group['PayCategoryGroupId'] : group['ShortName'] for group in group_data['Result']}
        

        data_list=[]
        for i, pay_category in enumerate(pay_categories_data):
            pay_cat_model=Pay_Category_Model()
            pay_cat_model.item_id = i+1
            pay_cat_model.pay_category_id = pay_category['PayCategoryId']
            pay_cat_model.processing_status = 'Pending'
            pay_cat_model.object_name=pay_category['ShortName']
            pay_cat_model.object_description=pay_category['LongName']
            pay_cat_model.pay_category_group=group_id_to_name.get(pay_category['PayCategoryGroupId'], '')
            pay_cat_model.multiplier_rate=pay_category['DefaultMultiplierRate']
            pay_cat_model.category='System' if pay_category['SystemRequired'] else 'Custom'
            pay_cat_model.show_hours=pay_category['ShowHours']
            pay_cat_model.show_amount=pay_category['ShowDollars']
            pay_cat_model.is_irregular_cost=pay_category['IsIrregularCost']
            pay_cat_model.sort_order=pay_category['SortOrder']
            pay_cat_model.code_name=pay_category['CodeName']
            pay_cat_model.reference_code=pay_category['XRefCode']
            pay_cat_model.reference_code_2=pay_category['XRefCode2']
            data_list.append(pay_cat_model)
            
        return data_list, group_id_to_name


In [ ]:
# Pay_Category_Reverse = Pay_Category_Reverse()
web_data, _ = Pay_Category_Reverse.Pay_Category_Reverse(driver)

# Fill Data in Sheet

In [137]:
from Service.DayForce.Utilites.gspread_helper import GspreadSheetHelper
from dotenv import load_dotenv
import os
load_dotenv()
class Pay_Category_fill_controller:
    GSH=GspreadSheetHelper()
    worksheet = GSH.getsheet(os.environ['SHEET_NAME'])

    # finding the start row and end row then clearing the data from the sheet
    start_row, end = GSH.start_end_row(worksheet)
    worksheet.batch_clear([f"A{start_row}:N{end}"])

    def pay_category_fill_controller(self, data):
        rows = []
        for i, pay_category in enumerate(data):
            rows.append([
                pay_category.item_id,
                pay_category.processing_status,
                pay_category.object_name,
                pay_category.object_description,
                pay_category.pay_category_group,
                pay_category.multiplier_rate,
                pay_category.category,
                pay_category.show_hours,
                pay_category.show_amount,
                pay_category.is_irregular_cost,
                pay_category.sort_order,
                pay_category.code_name,
                pay_category.reference_code,
                pay_category.reference_code_2
            ])

        self.GSH.update_sheet(self.worksheet,'A','N',rows)
            

In [138]:
fill_controller = Pay_Category_fill_controller()
fill_controller.pay_category_fill_controller(web_data)

# Loader Function

In [139]:
from Service.DayForce.Utilites.gspread_helper import GspreadSheetHelper
class Pay_Category_loader_controller:
    GSH=GspreadSheetHelper()
    worksheet = GSH.getsheet(os.environ['SHEET_NAME'])

    def pay_category_loader_controller(self):
        start_row, end = self.GSH.start_end_row(self.worksheet)
        data = self.worksheet.get(f"A{start_row}:N{end}",pad_values=True)
        pay_category_models = []
        row_count = 0
        for row in data:
            if row[1] == 'Processed':
                continue
            row_count += 1
            pay_cat_model=Pay_Category_Model()
            pay_cat_model.item_id=row[0]
            pay_cat_model.object_name=row[2]
            pay_cat_model.object_description=row[3]
            pay_cat_model.pay_category_group=row[4]
            pay_cat_model.multiplier_rate=row[5]
            pay_cat_model.category=row[6]
            pay_cat_model.show_hours=row[7]
            pay_cat_model.show_amount=row[8]
            pay_cat_model.is_irregular_cost=row[9]
            pay_cat_model.sort_order=row[10]
            pay_cat_model.code_name=row[11]
            pay_cat_model.reference_code=row[12]
            pay_cat_model.reference_code_2=row[13]
            
            pay_category_models.append(pay_cat_model)
            
        return pay_category_models

In [140]:
loader_controller = Pay_Category_loader_controller()
sheet_data = loader_controller.pay_category_loader_controller()

# Validation

In [58]:
from gspread_formatting import format_cell_ranges
from Service.DayForce.Utilites.gspread_helper import GspreadSheetHelper
class Pay_Category_Validation:
    def validate_pay_category(self, web_models):
        GSH=GspreadSheetHelper()
        worksheet = GSH.getsheet(os.environ['SHEET_NAME'])
        loader_controller = Pay_Category_loader_controller()
        sheet_models = loader_controller.pay_category_loader_controller()
        
        helper=automate_helper()
        green ,red = GSH.colors_r_g()
        formate_list=[]
        for i in range(len(web_models)):
            if web_models[i].object_name == sheet_models[i].object_name :
                color= green
                formate_list.append((f"C{i+4}", color))

                color = green if str(web_models[i].object_description) == helper.maps(sheet_models[i].object_description) else red
                formate_list.append((f"D{i+4}", color))
                
                color = green if str(web_models[i].pay_category_group) == sheet_models[i].pay_category_group else red
                formate_list.append((f"E{i+4}", color))

                color = green if web_models[i].multiplier_rate == float(sheet_models[i].multiplier_rate) else red
                formate_list.append((f"F{i+4}", color))

                color = green if web_models[i].category == helper.maps(sheet_models[i].category) else red
                formate_list.append((f"G{i+4}", color))

                color = green if str(web_models[i].show_hours) == helper.maps(sheet_models[i].show_hours.capitalize()) else red
                formate_list.append((f"H{i+4}", color))

                color = green if str(web_models[i].show_amount) == helper.maps(sheet_models[i].show_amount.capitalize()) else red
                formate_list.append((f"I{i+4}", color))

                color = green if str(web_models[i].is_irregular_cost) == helper.maps(sheet_models[i].is_irregular_cost.capitalize()) else red
                formate_list.append((f"J{i+4}", color))

                color = green if str(web_models[i].sort_order) == helper.maps(sheet_models[i].sort_order) else red
                formate_list.append((f"K{i+4}", color))

                color = green if str(web_models[i].code_name) == helper.maps(sheet_models[i].code_name) else red
                formate_list.append((f"L{i+4}", color))

                color = green if str(web_models[i].reference_code) == helper.maps(sheet_models[i].reference_code) else red
                formate_list.append((f"M{i+4}", color))

                color = green if str(web_models[i].reference_code_2) == helper.maps(sheet_models[i].reference_code_2) else red
                formate_list.append((f"N{i+4}", color))
            else:
                color = red
                formate_list.append((f"C{i+4}", color))
                formate_list.append((f"D{i+4}", color))
                formate_list.append((f"E{i+4}", color))
                formate_list.append((f"F{i+4}", color))
                formate_list.append((f"G{i+4}", color))
                formate_list.append((f"H{i+4}", color))
                formate_list.append((f"I{i+4}", color))
                formate_list.append((f"J{i+4}", color))
                formate_list.append((f"K{i+4}", color))
                formate_list.append((f"L{i+4}", color))
                formate_list.append((f"M{i+4}", color))
                formate_list.append((f"N{i+4}", color))

        # length = len(formate_list)//2
        # print("formating first half")
        # format_cell_ranges(worksheet,formate_list[:length])
        format_cell_ranges(worksheet,formate_list[:1000])
        print("formating second half")
        # format_cell_ranges(worksheet,formate_list[length:])
        print("Validation Completed")
        


In [59]:
val = Pay_Category_Validation()
val.validate_pay_category(web_data)

formating second half
Validation Completed


# Automation

In [208]:
status=[["Processed"]]
status=status*10

print(status)

[['Processed'], ['Processed'], ['Processed'], ['Processed'], ['Processed'], ['Processed'], ['Processed'], ['Processed'], ['Processed'], ['Processed']]


In [141]:
class automate_helper:
    def helper(self, data_sheet , groups,driver,id):

        rev_groups={k:v for v,k in groups.items()}

        payload=[{"PayCategoryId": id,
        "ShortName": data_sheet.object_name,
        "LongName": data_sheet.object_description,
        "ClientId": 112075,
        "CultureId": None,
        "XRefCode": data_sheet.reference_code,
        "XRefCode2": data_sheet.reference_code_2,
        "PayCategoryGroupId": rev_groups.get(data_sheet.pay_category_group),
        "SortOrder": data_sheet.sort_order,
        "DefaultMultiplierRate": data_sheet.multiplier_rate,
        "SystemRequired": True if data_sheet.category == 'System' else False,
        "CodeName": data_sheet.code_name,
        "ShowHours": True if data_sheet.show_hours == 'TRUE' else False,
        "ShowDollars": True if data_sheet.show_amount == 'TRUE' else False,
        "IsIrregularCost": True if data_sheet.is_irregular_cost == 'TRUE' else False,
        "CollapsableLabel": None,
        "NodeLevel": None,
        "CurrentClientId": None,
        "CurrentClientName": None,
        "NumberOfChild": None,
        "ClientEntityId": "89aa80fb-335d-43a2-87d6-f951531ef970",
        "EntityState": 2,
        "LastModifiedTimestamp": None,
        "OriginalValues": None,
        "ExtendedProperties": []}]
        call_update_api=Pay_Category_Get_Post()
        call_update_api.update_pay_category(driver, payload)
    
    def maps(self,data):
        return 'None' if data == '' else data

In [ ]:
from Service.DayForce.Utilites.selenium_helper import handywrapper
class Pay_Category_Automate:
    def automate_pay_category(self):
        loader_controller = Pay_Category_loader_controller()
        sheet_models = loader_controller.pay_category_loader_controller()
        client_id=0
        wrapper = handywrapper(driver)
        Pay_Category = Pay_Category_Reverse()
        web_models, groups = Pay_Category.Pay_Category_Reverse(driver)
        helper = automate_helper()
        for i in range(len(sheet_models)):
            id = int(sheet_models[i].item_id)-1
            if web_models[id].object_name == sheet_models[i].object_name :



                
                similarity=(str(web_models[id].object_description) == helper.maps(sheet_models[i].object_description) and 
                        str(web_models[id].pay_category_group) == sheet_models[i].pay_category_group and 
                        web_models[id].multiplier_rate == float(sheet_models[i].multiplier_rate) and 
                        str(web_models[id].category) == helper.maps(sheet_models[i].category) and 
                        str(web_models[id].show_hours) == helper.maps(sheet_models[i].show_hours.capitalize()) and 
                        str(web_models[id].show_amount) == helper.maps(sheet_models[i].show_amount.capitalize()) and 
                        str(web_models[id].is_irregular_cost) == helper.maps(sheet_models[i].is_irregular_cost.capitalize()) and
                        str(web_models[id].sort_order) == helper.maps(sheet_models[i].sort_order) and 
                        str(web_models[id].code_name) == helper.maps(sheet_models[i].code_name) and 
                        str(web_models[id].reference_code) == helper.maps(sheet_models[i].reference_code) and 
                        str(web_models[id].reference_code_2) == helper.maps(sheet_models[i].reference_code_2) )
            else:
                print("No pay category found for object so, creating new one:", sheet_models[i].object_name)
                client_id+=1
                similarity=False

            if not similarity:
                print("Automating for Pay Category Object Name:", sheet_models[i].object_name)
                helper.helper(sheet_models[i], groups, driver,len(web_models)+client_id)

        wrapper.click_element('XPATH', "//span[@class='dijit dijitReset dijitInline dijitButton' and @widgetid = 'UI_Mixins__TooltipMixin_5']")


In [ ]:
automate = Pay_Category_Automate()
automate.automate_pay_category()

Automating for Pay Category Object Name: $etuping


In [ ]:
['hello','desc',['hello','1','url1'],['hello','2','url2'],['hello','3','url3'],['hello','4','url4']],
['hello2','desc',['hello2','1','url1'],['hello2','2','url2'],['hello2','3','url3'],['hello2','4','url4']],
